In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

# Safe CSV loading with validation
def load_csv_safely(file_path, required_columns=None):
    """
    Safely load CSV file with input validation.
    Args:
        file_path: Path to CSV file
        required_columns: List of required column names
    Returns:
        DataFrame if valid, raises ValueError otherwise
    """
    # Validate and resolve path to prevent path traversal
    file_path = Path(file_path).resolve()
    
    # Only allow files in current directory and subdirectories
    if not file_path.exists():
        raise ValueError(f"File not found: {file_path}")
    
    # Load CSV with security settings
    df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    
    # Validate required columns exist
    if required_columns:
        missing = set(required_columns) - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
    
    # Validate no empty dataframe
    if df.empty:
        raise ValueError("CSV file is empty")
    
    return df

# Load data safely
df = load_csv_safely(
    "software_assets_data.csv",
    required_columns=['software_version', 'software_compatbile', 'other_software_Version', 'other_sw_version_defect_count']
)

# Function to apply Pareto analysis - reusable for any item and value column
def apply_pareto_analysis(data, item_column, value_column, pareto_threshold=80):
    """
    Apply Pareto analysis and identify contributors to target percentage.
    Returns a dict mapping items to whether they're Pareto contributors.
    """
    if not isinstance(pareto_threshold, (int, float)) or not 0 < pareto_threshold <= 100:
        raise ValueError("pareto_threshold must be between 0 and 100")
    
    aggregated = data.groupby(item_column)[value_column].sum().sort_values(ascending=False)
    cumsum = aggregated.cumsum()
    cumulative_pct = (cumsum / cumsum.iloc[-1]) * 100
    return (cumulative_pct <= pareto_threshold).to_dict()

# Function to generate recommendations
def generate_recommendations(df):
    """Generate upgrade/decommission recommendations based on Pareto analysis."""
    
    # Apply Pareto analysis for other software versions
    pareto_contributors = apply_pareto_analysis(
        df, 
        item_column="other_software_Version", 
        value_column="other_sw_version_defect_count"
    )
    
    # Create final recommendations using vectorized operations
    recommendations_df = pd.DataFrame({
        'Software Version': df['software_version'],
        'Recommended Action': df.apply(
            lambda row: 'Upgrade' 
            if row['software_compatbile'] == "Yes" or not pareto_contributors.get(row['other_software_Version'], False)
            else 'Decommission',
            axis=1
        )
    })
    
    return recommendations_df.to_dict('records')

# Generate recommendations with error handling
try:
    recommendations = generate_recommendations(df)
    
    # Output the recommendations
    for rec in recommendations:
        print(f"Software: {rec['Software Version']} | Action: {rec['Recommended Action']}")
except Exception as e:
    print(f"Error generating recommendations: {e}")

Software: Java 6 Action: Upgrade
Software: Java 6 Action: Upgrade
Software: Java 7 Action: Upgrade
Software: Java 7 Action: Upgrade
Software: Java 8 Action: Decommission
Software: Java 8 Action: Upgrade
Software: Java 8 Action: Upgrade
Software: Java 8 Action: Upgrade
Software: Java 9 Action: Decommission
Software: Java 9 Action: Decommission
Software: Java 9 Action: Decommission
Software: Java 10 Action: Decommission
Software: Java 10 Action: Decommission
Software: Java 10 Action: Decommission
Software: Java 11 Action: Upgrade
Software: Java 11 Action: Upgrade
Software: Java 11 Action: Upgrade
Software: Java 11 Action: Upgrade
Software: Java 12 Action: Upgrade
Software: Java 12 Action: Upgrade
Software: Java 13 Action: Upgrade
Software: Java 13 Action: Upgrade
Software: Java 14 Action: Upgrade
Software: Java 14 Action: Upgrade
Software: Java 15 Action: Upgrade
Software: Java 15 Action: Upgrade
Software: Java 16 Action: Upgrade
Software: Java 16 Action: Upgrade
Software: Java 17 Action

# New Section